In [65]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder

In [66]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [67]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [68]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [69]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [70]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [71]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [72]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

- model

In [73]:
class MyNN():
    def __init__(self,X):
        self.weights = torch.rand(X.shape[1],1,dtype=torch.float64,requires_grad=True)
        self.bias = torch.zeros(1,dtype=torch.float64,requires_grad=True)

    def forward(self,X):
        z = torch.matmul(X,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_fuction(self,y_pred,y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred,epsilon,1-epsilon)

        loss = -(y_train_tensor * torch.log(y_pred) + (1- y_train_tensor)*torch.log(1-y_pred)).mean()
        return loss

In [74]:
learning_rate = 0.1
epochs =25

In [75]:
# create model
model = MyNN(X_train_tensor)

for i in range(epochs):
        # forward pass
        y_pred =  model.forward(X_train_tensor)

        # Loss 
        loss = model.loss_fuction(y_pred,y_train_tensor)

        # backward
        loss.backward()

        # update parameters
        with torch.no_grad():
                model.weights -= learning_rate * model.weights.grad 
                model.bias -= learning_rate * model.bias.grad 

        #zero Gradient
        model.weights.grad.zero_()
        model.bias.grad.zero_()

        # loss in each epochs
        print(f'Epoch: {i+1},loss: {loss.item()}')


Epoch: 1,loss: 3.9944770632797226
Epoch: 2,loss: 3.8786313563856702
Epoch: 3,loss: 3.7564648509154193
Epoch: 4,loss: 3.6240206918217224
Epoch: 5,loss: 3.487181131571454
Epoch: 6,loss: 3.3506692363302015
Epoch: 7,loss: 3.212045511960849
Epoch: 8,loss: 3.0635130276103366
Epoch: 9,loss: 2.910036771484676
Epoch: 10,loss: 2.7501258259398
Epoch: 11,loss: 2.584655703187727
Epoch: 12,loss: 2.4108304341682736
Epoch: 13,loss: 2.236226100076545
Epoch: 14,loss: 2.058995129075507
Epoch: 15,loss: 1.8876654606367786
Epoch: 16,loss: 1.7232271229543477
Epoch: 17,loss: 1.5620162809856935
Epoch: 18,loss: 1.4096822322207558
Epoch: 19,loss: 1.272050718652487
Epoch: 20,loss: 1.1519224618100026
Epoch: 21,loss: 1.0511805804761305
Epoch: 22,loss: 0.9705963853966163
Epoch: 23,loss: 0.9089319087464341
Epoch: 24,loss: 0.8632226933836822
Epoch: 25,loss: 0.8300254981531714


- evalution

In [76]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(accuracy)


tensor(0.5351)
